# RAG Pipeline using Groq, ChromaDB and Sentence Transformers

This notebook is designed to run from the project repository. Place the downloaded PDF files inside `RAG_pdfs/` and keep your API key in a local `.env` file.

In [ ]:
# Optional text-file test
path = 'Python.txt'

with open(path, 'r', encoding='utf-8') as file:
    text = file.read()

print(text)

In [ ]:
# Install project dependencies
%pip install -r requirements.txt

#1. Load Lib.

In [ ]:
import langchain
import langchain_core
import langchain_community
import pypdf
import pymupdf
import sentence_transformers
from langchain_core import documents
from langchain_community.document_loaders.pdf import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
import os
import chromadb      #Creating Vector Databse
import uuid          #Generating Random unique id for the db
from sklearn.metrics.pairwise import cosine_similarity
from langchain_groq import ChatGroq


print("All LangChain/RAG libraries imported successfully!")

#2. Load the Documents

In [ ]:
# Load PDF files from the project dataset folder
pdfs_folder = 'RAG_pdfs'

def load_pdfs():
  path = pdfs_folder
  pdfs_count = 0
  pdfs_doc = []

  if not os.path.exists(path):
    raise FileNotFoundError(
        f"PDF folder not found: {path}. Add the dataset PDFs to RAG_pdfs/ before running this section."
    )

  for item in os.listdir(path):
    if item.lower().endswith('.pdf'):
      pdf_path = os.path.join(path, item)
      pdf_loader = PyMuPDFLoader(pdf_path)
      pdf_doc = pdf_loader.load()
      pdfs_doc.extend(pdf_doc)
      pdfs_count += 1

  print("Total PDF Documents count: ", pdfs_count)
  print("Total Documents Page count: ", len(pdfs_doc))
  return pdfs_doc

In [ ]:
pdf_dt = load_pdfs()

In [ ]:
print(type(pdf_dt[2]))

#3. Partition Documents into Chunks

In [ ]:
def split_into_chunks(documents, chunk_size=500, chunk_overlap=50):
  textSplitter = RecursiveCharacterTextSplitter(
      chunk_size=chunk_size,
      chunk_overlap=chunk_overlap
  )

  pdfs_chunks = textSplitter.split_documents(documents)

  return pdfs_chunks

In [ ]:
pdf_chunk = split_into_chunks(pdf_dt)

print("Total counts of Chunks: ", len(pdf_chunk))
print("Avg. no. chunks per page: ", len(pdf_chunk)/len(pdf_dt))

#4. Pass n number of Chunks through n number of Embedding module(Sentence-Transformer), convert each into dense Vector

In [ ]:
class EmbeddingPerformer:
  embedding_model = 'all-MiniLM-L6-v2'

  # Constructor
  def __init__(self, model_name=embedding_model):
    #Initializing the selected model
    self.embedding_model=model_name
    print("Loading Embedding Model as", self.embedding_model)

    #Creating that model object
    self.model = SentenceTransformer(self.embedding_model)
    print("Embedding Dimension: ", self.model.get_sentence_embedding_dimension())

  # Apply Selected Embedding model on Chunks and convert into dense vectors
  def apply_embedding(self, chunk_dt):
    embeddings = self.model.encode(chunk_dt, show_progress_bar=True)
    print("Embedding Shape: ", embeddings.shape)
    return embeddings


In [ ]:
embedding_performer = EmbeddingPerformer()

#5. Insert all Vectors into VectorDB

In [ ]:
# Instead of VectorDB, creating locally vectorStore
class VectorStoreGenerator:
  vectorStore_path = 'vector_store'

  # Contructor
  # Create a Client --> that client will create a collection framework --> that will store collections as data
  # This whole data of collections will store in vectorStore
  def __init__(self, persist_directory=vectorStore_path, collection_name='pdf_doc'):
    self.persist_directory = persist_directory
    self.collection_name = collection_name
    self.collection = None
    self.client = None
    self._create_vectotStore()

  # Private fnx for creating the store, which have client & collections
  def _create_vectotStore(self):
    # step1: make store dir
    if not os.path.exists(self.persist_directory):
      os.makedirs(self.persist_directory)

    # step2: Create the client for requesting to store and responsing back from store
    self.client = chromadb.PersistentClient(path=self.persist_directory)

    # step3: Create collection framework to store collections
    self.collection = self.client.get_or_create_collection(
        name = self.collection_name,
        metadata = {
            "hnsw:space": "cosine",
            "desc": "vector store collections for embedding of pdf doc. in RAG pipeline."
        }
    )

    print("Vector Store Collection name: ", self.collection_name)
    print("Number of the collections: ", self.collection.count())

  # Insert Dataset into Vector Store
  def add_data(self, chunks, embeddings):
    #Check for n number of Chunks should have n number of embeddings
    if len(chunks) != len(embeddings):
      raise ValueError("Number of Chunkes should be equal to number of Embeddings.")

    #Deafult variables to store all features of table inside store
    ids = []
    metadatas = []
    dataset = []
    embedding_dt = []

    # To work on different datatypes, zip into dataset
    for i, (doc, embedding) in enumerate(zip(chunks, embeddings)):
      #Feature1: Generate random ids and store
      doc_id = f"doc_{uuid.uuid4()}"
      ids.append(doc_id)

      #Feature2: Generate metadata(index, size of document, metadata content) and store
      doc_metadata = dict(doc.metadata)
      doc_metadata["doc_index"] = i
      doc_metadata["doc_size"] = len(doc.page_content)
      metadatas.append(doc_metadata)

      #Feature3: Generate the document structure of all chunks
      dataset.append(doc.page_content)

      #Feature4: Store all the embeddings of every chunks
      embedding_dt.append(embedding.tolist())

    #Store all features into collection of ChormaDB
    self.collection.add(
        ids=ids,
        metadatas=metadatas,
        documents=dataset,
        embeddings=embedding_dt
    )
    print(f"{len(chunks)} chunks successfully added to Vector Store.")

In [ ]:
vectorSt = VectorStoreGenerator()

#6. Activate Ingestion Pipeline of RAG

In [ ]:
# Extract data from every chunk inside all chunk dataset
chunk_content = [chunk.page_content for chunk in pdf_chunk]

# Convert all chunks into embeddings
embeddings = embedding_performer.apply_embedding(chunk_content)

# Store all chunks and corresponding embeddings into Vector Store
vectorSt.add_data(pdf_chunk, embeddings)

In [ ]:
# embedding matrix has:

# 699 rows → one embedding vector for each of your 699 chunks
# 384 columns → each chunk is represented by 384 numerical values
# They are numerical features/dimensions representing the semantic information of that chunk.


# So the total number of numerical values are:
# 699 × 384 = 268,416 values

#                 384 dimensions
#           ┌───────────────────────┐
# Chunk 1   │ x₁ x₂ x₃ ... x₃₈₄     │
# Chunk 2   │ x₁ x₂ x₃ ... x₃₈₄     │
# Chunk 3   │ x₁ x₂ x₃ ... x₃₈₄     │
#   ...     │          ...          │
# Chunk 699 │ x₁ x₂ x₃ ... x₃₈₄     │
#           └───────────────────────┘
#               699 embeddings

#7. RAG Retriever Pipeline

In [ ]:
class RAGRetriever:
  # To retrieve data, want Embedded Data & VectorStore Embedded data
  def __init__(self, embedding_performer, vectorSt):
    self.embedding_performer = embedding_performer
    self.vectorStore_generator = vectorSt

  # Retrieve on the basis of User input, Top k sementic similarity val upon min sementic score limit(threshold)
  def retrieve(self, query, top_k=5, score_threshold=0.1):
    # Convert User query val into embedded dataset
    query_embedded = self.embedding_performer.apply_embedding([query])[0]   #query as argument will be in list formate

    # Sementic search between Query embedded data & Vector Store data stored in collection framework
    sementic_result = self.vectorStore_generator.collection.query(
        query_embeddings = [query_embedded.tolist()],
        n_results = top_k
    )

    #Store retrieve documents
    retrieve_docs = []

    #Check for Similarity Score val within min threshold val
    #Check for valid data
    if sementic_result['documents'] and sementic_result['documents'][0]:
      ids = sementic_result['ids'][0]
      metadatas = sementic_result['metadatas'][0]
      documents = sementic_result['documents'][0]
      distances = sementic_result['distances'][0]

      for i, (doc_id, metadata, doc, distance) in enumerate(zip(ids, metadatas, documents, distances)):
        # Relation between distance & similarity val
        similarity_score = 1 - distance

        # If valid score, then only return as retrieve documents
        if similarity_score >= score_threshold:
          retrieve_docs.append({
              "id": doc_id,
              "metadata": metadata,
              "document": doc,
              "distance": distance,
              "similarity_score": similarity_score,
              "rank": i + 1                        # top k by ranking
          })
          print(f"Rank: {i + 1} | Similarity Score: {similarity_score:.4f}")

      print(f"Count of retrieval documents {len(retrieve_docs)}")
    #If not valid documents
    else:
      print("No documents found for retrieval.")

    return retrieve_docs

In [ ]:
# Object for RAG Retrieval class
rag_retrieve = RAGRetriever(embedding_performer, vectorSt)

In [ ]:
# Test in a sample data
rag_retrieve.retrieve("What is RAG", top_k=5)

#8. Augment DataSet & Intregret on Pre-build LLMs

In [ ]:
!pip install python-dotenv

In [ ]:
from dotenv import load_dotenv

# Load the local .env file. Never commit .env to GitHub.
load_dotenv()

Groq_API_key = os.getenv("Groq_API_key")

if not Groq_API_key:
    raise ValueError("Groq_API_key not found. Create a .env file with Groq_API_key=your_api_key")

print("API Key loaded: True")

In [ ]:
# Create object to use LLMs(Groq)

llms_groq = ChatGroq(
    groq_api_key = Groq_API_key,         # Generated own Groq LLMs access key
    model_name = "openai/gpt-oss-120b",  # Insert the latest version
    temperature = 0.1,                    # Creativity, Imagination limit Increase == Factual limit decreases
    max_tokens = 1024                    # In every generated response should use tokens within this limit
)

In [ ]:
# Generate RAG retrieval-augmented output
def generate_output(query, rag_retrieve, llms_groq, top_k=5):
  # Augmentation between Retrieval data + User query
  results = rag_retrieve.retrieve(query, top_k=top_k)

  # Valid result represented as Context
  context = "\n".join(
      chunk["document"]
      for chunk in results
  ) if results else ""
  # Not valid resullt --> empty string. Instead of that show a msg.
  if not context:
    return "I could not find relevant information in the provided documents."

  # Prompt = Valid msg + Valid Context + User Org. query
  llm_prompt = f"""
  You are a question-answering assistant for a RAG system.
  Answer the user's question using ONLY the information provided in the context.
  Do not use your own knowledge to answer the question.
  Context:
  {context}
  Question:
  {query}
  Answer:
  """

  #Generate the final response
  response = llms_groq.invoke(llm_prompt)

  #Output will be only content part of the response
  print("Token usage details of RAG request: \n",response.usage_metadata)
  return response.content, results

In [ ]:
import pandas as pd

queries = [
    "What is RAG?",
    "What is Encoder-Decoder?",
    "What is Python?",
    "Full form of NLP?",
    "Who is Kelvin Guu?",
    "Use of Open-domain question answering (Open-QA)?",
    "How Knowledge Retriever is defined?",
    "What is DPR?",
    "Who is Payal Bajaj?",
    "Two References name of Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks?",
    "Comparison between the three paradigms of RAG.",
    "Comparison image between the three paradigms of RAG.",
    "What happened in 2014?",
    "The unfiltered version of TriviaQA is used for open-domain question answering. This Statement is True or False",
    "LLM means, Large Language Models. This Statement is True or False?"
]

results_table = []

for query_no, query in enumerate(queries, start=1):
    output, retrieved_results = generate_output(
        query,
        rag_retrieve,
        llms_groq,
        top_k=5
    )

    retrieval_count = len(retrieved_results)

    # Use the similarity score of the highest-ranked retrieved document.
    similarity_score = (
        round(retrieved_results[0]["similarity_score"], 4)
        if retrieved_results else 0.0
    )

    results_table.append({
        "Query No.": query_no,
        "Query": query,
        "Count of Retrieval Documents": retrieval_count,
        "Similarity Score": similarity_score,
        "Output": output
    })

# Create and display the final evaluation table.
results_df = pd.DataFrame(results_table)
display(results_df)

# Save results in the repository root.
csv_path = "RAG_Query_Results.csv"
results_df.to_csv(csv_path, index=False, encoding="utf-8")

print(f"Results successfully saved to: {csv_path}")